In [ ]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from pathlib import Path
from typing import Dict, List
import seaborn as sns
import re
from collections import defaultdict

In [ ]:
# Helper maps / constants

region_mapping = {
    "0": "FRA",
    "1": "SF",
    "2": "BAN",
    "3": "SYD"
}

# Plotting
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

output_base_path = Path("output")

In [ ]:
# Load data
benchmark_dir = "data"
benchmark_name = "test"
file_pattern = f"{benchmark_name}_*.csv"

files = glob.glob(f"{benchmark_dir}/{benchmark_name}/{file_pattern}")

# dfs = []
# for f in files:
#     region_num = f.split("_")[-1].split(".")[0]
#     df = pd.read_csv(f)
#     mapping = region_mapping[region_num]
#     if mapping is None:
#         print(f"Mapping for region number {region_mapping} doesn"t exist!")
#     df["region"] = mapping
#     dfs.append(df)
#
# data = pd.concat(dfs, ignore_index=True)
# data["region"] = data["region"].astype("string")

In [ ]:
# print(data.info())
# print(data.describe())
#
# print(data["region"].value_counts())

In [ ]:
# plt.figure(figsize=(10, 6))
# for region, df_region in data.groupby("region"):
#     plt.hist(
#         df_region["latency"],
#         bins=30,
#         alpha=0.5,
#         label=f"Region {region}"
#     )
#
# plt.xlabel("Latency (ms)")
# plt.ylabel("Frequency")
# plt.title("Latency Distribution per Region")
# plt.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
N = 1000   # number of items (ranks)
alpha = 1.1  # Zipf exponent

# Generate Zipfian-like data
ranks = np.arange(1, N + 1)
freqs = 1 / ranks**alpha
freqs /= freqs.sum()  # normalize

# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(ranks, freqs, color="royalblue")
axes[0].set_title("Zipfian Distribution (Linear Scale)")
axes[0].set_xlabel("Rank")
axes[0].set_ylabel("Frequency (normalized)")
axes[0].grid(True, linestyle="--", linewidth=0.5)

axes[1].loglog(ranks, freqs, marker="o", markersize=3,
               linestyle="none", color="darkorange")
axes[1].set_title("Zipfian Distribution (Log–Log Scale)")
axes[1].set_xlabel("Rank (log)")
axes[1].set_ylabel("Frequency (log)")
axes[1].grid(True, which="both", linestyle="--", linewidth=0.5)

plt.tight_layout()
plt.show()

# YCSB Evaluation

## Directory structure

- Base directory: data
- Each benchmark run is stored in a separate directory and has the prefix `ycsb_`
  - All benchmark files in the directory have the prefix `ycsb_`, then `l_` if
  it was the load phase, or `t_` if it was the transaction phase.
  - After that, the benchmark name is added. By convention, it is 
    `{user defined name}-{transaction count}-{run number}`. For example,
    `trivial-3000-4` is the 4th run of the `trivial` configuration of ISOS, with
    3000 operations per client replica, i.e., 12000 operations total with 4
    client nodes.
  - After that, the client node ID is appended as `_{client id}`, ranging from
    `0`-`3` for the transaction nodes. The client for the load phase has the ID `5`.
  - Each client saves a `.csv` and `.raw` file. The CSV file is the default YCSB
    human-readable overview. As we do the evaluation separately in Python, we
    need the latency from each request, as the `timeseries` or `histogram` are
    not as detailed / granular. The `raw` file is formatted as CSV.
- Example files:
  - Benchmark name `trivial-raw-3000-2`, i.e., the second run of trivial-raw
  with 3000 operations. Files from the YCSB transaction phase on client node 3:
    - CSV file: `data/ycsb_trivial-raw-3000-2/ycsb_t_trivial-raw-3000-2_3.csv`
    - RAW file: `data/ycsb_trivial-raw-3000-2/ycsb_t_trivial-raw-3000-2_3.raw`
    - For analysis of all runs of all clients, add `trivial-raw-3000` to the
    `ycsb_raw_benchmarks` array
  
## Evaluation

- If a benchmark has multiple runs, all runs are considered for the evaluation.
- Only the `.raw` files are evaluated. The `.csv` files are for 
- `READ` and `UPDATE` commands are combined, `CLEANUP` commands are ignored
- We generate the following graphs for each benchmark:
  - Overview of performance: Average overall client latency (ms) and throughput
    (ops/s) across all benchmark runs and all clients for the given benchmark
  - Latency percentiles (p1, p5, p50, p90, p95, p99, p99.9) overall, with
    percentile bands
  - Latency percentiles split by client
  - Average overall latency over (benchmark) time, in 1 second buckets


In [ ]:
# List of benchmarks with raw files
ycsb_raw_benchmarks = [
    "trivial-raw-3000",
]

def find_benchmark_files(
    data_dir: str, 
    benchmark_name: str
) -> Dict[int, List[Path]]:
    """
    Find all raw files for a benchmark pattern across all runs and clients.
    
    Returns: {run_number: [client_0_path, client_1_path, ...]}
    """
    
    # First, get directories of all runs
    # Pattern: ycsb_{benchmark_pattern}-{run_number}
    base_data_dir = Path(data_dir)
    runs = defaultdict(list)
    run_paths = base_data_dir.glob(f"ycsb_{benchmark_name}-*")
    for run_dir in run_paths:
        # Extract run number from directory name
        match = re.search(rf"{benchmark_name}-(\d+)$", run_dir.name)
        if not match:
            continue
        
        run_num = int(match.group(1))
        
        # Find all client files (0-3) for this run

        for client_id in range(4):
            pattern = f"ycsb_t_{benchmark_name}-*_{client_id}.raw"
            client_files = list(run_dir.glob(pattern))
            
            if client_files:
                runs[run_num].append((client_id, client_files[0]))
    
    return dict(runs)

def parse_raw_file(
    filepath: Path, 
    client_id: int, 
    run_number: int
) -> pd.DataFrame:
    """Parse a YCSB raw file and add metadata"""
    data = []
    
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if (not line or line.startswith("#") or 
                "latency raw data" in line.lower()):
                continue
            
            parts = line.split(",")
            if len(parts) == 3:
                op, timestamp, latency = parts
                data.append({
                    "operation": op,
                    "absolute_timestamp_ms": int(timestamp),
                    "latency_us": int(latency),
                    "client_id": client_id,
                    "run_number": run_number
                })
    
    df = pd.DataFrame(data)
    if len(df) > 0:
        # Remove "CLEANUP" operations from dataframe
        df = df[df["operation"] != "CLEANUP"]

        # Convert data to more useful units
        # Latency in ms
        df["latency_ms"] = df["latency_us"] / 1000
        df = df.drop(columns=["latency_us"])
        # Relative timestamp of the operation in the replica, starting from the
        # first measurement of a run
        start_time = df["absolute_timestamp_ms"].min()
        df["rel_timestamp_ms"] = (
            df["absolute_timestamp_ms"] - start_time
        ) / 1000 

    
    return df


def load_benchmark_data(
    data_dir: str, 
    benchmark_pattern: str
) -> pd.DataFrame:
    """
    Load all data for a benchmark pattern across all runs and clients, and then
    do som
    
    Args:
        data_dir: Base data directory
        benchmark_pattern: Pattern like "trivial-raw-3000"
    
    Returns:
        Combined DataFrame with all runs and clients
    """
    runs = find_benchmark_files(data_dir, benchmark_pattern)
    
    if not runs:
        raise ValueError(
            f"No data found for benchmark pattern: {benchmark_pattern}"
        )
    
    all_dfs = []
    
    for run_num, client_files in sorted(runs.items()):
        print(f"Loading run {run_num}...")
        for client_id, filepath in sorted(client_files):
            print(f"  - Client {client_id}: {filepath.name}")
            df = parse_raw_file(filepath, client_id, run_num)
            all_dfs.append(df)
    
    # Combine all data
    combined_df = pd.concat(all_dfs, ignore_index=True)
    
    # For each run, calculate the 
    combined_df["overall_run_ms"] = combined_df.groupby("run_number")[
        "absolute_timestamp_ms"
    ].transform(lambda x: (x - x.min()))
    
    print(f"\nLoaded {len(combined_df):,} operations")
    print(f"  - {combined_df["run_number"].nunique()} runs")
    print(f"  - {combined_df["client_id"].nunique()} clients per run")
    
    return combined_df

# Testing
test_df = load_benchmark_data("data", "trivial-raw-3000")
print(test_df.describe())
print(test_df.head(3))

print(test_df.sort_values(by="overall_run_ms").head(20))

def first_request_per_client(df: pd.DataFrame) -> pd.DataFrame:
    """Return the first observed request for each (run_number, client_id)."""

    # idx of the row with minimum rel_timestamp_ms per (run_number, client_id)
    idx = df.groupby(["run_number", "client_id"])["rel_timestamp_ms"].idxmin()
    # select useful columns and rename
    firsts = df.loc[idx, ["run_number","client_id","absolute_timestamp_ms"]].copy()

    # Compute per-run minimum absolute timestamp 
    run_min_map = df.groupby("run_number")["absolute_timestamp_ms"].min().to_dict()
    firsts["diff_after_initial_ms"] = firsts["absolute_timestamp_ms"] - firsts["run_number"].map(run_min_map)

    firsts = firsts.sort_values(["run_number","diff_after_initial_ms"]).reset_index(drop=True)
    return firsts

firsts = first_request_per_client(test_df)
print(firsts)


# Data analysis

def calculate_overall_statistics(df: pd.DataFrame) -> Dict:
    """Calculate statistics across all runs and clients"""
    
    # Overall statistics
    total_ops = len(df)
    latencies = df["latency_ms"]
    
    # Per-run statistics for duration and throughput
    run_stats = []
    for run_num in df["run_number"].unique():
        run_df = df[df["run_number"] == run_num]
        duration_s = (
            run_df["absolute_timestamp_ms"].max() - run_df["absolute_timestamp_ms"].min()
        ) / 1000
        throughput = len(run_df) / duration_s if duration_s > 0 else 0
        run_stats.append({
            "run": run_num,
            "duration_s": duration_s,
            "throughput": throughput,
            "operations": len(run_df)
        })
    
    run_stats_df = pd.DataFrame(run_stats)
    
    # Percentiles
    percentiles = [1, 5, 50, 90, 95, 99, 99.9]
    
    stats = {
        "total_operations": total_ops,
        "total_runs": df["run_number"].nunique(),
        "total_clients": df["client_id"].nunique(),
        "avg_duration_s": run_stats_df["duration_s"].mean(),
        "avg_throughput_ops_s": run_stats_df["throughput"].mean(),
        "throughput_std": run_stats_df["throughput"].std(),
        "avg_latency_ms": latencies.mean(),
        "latency_std_ms": latencies.std(),
        "min_latency_ms": latencies.min(),
        "max_latency_ms": latencies.max(),
        "percentiles": {
            f"p{p}": np.percentile(latencies, p) 
            for p in percentiles
        },
        "run_details": run_stats_df
    }
    
    return stats


calculate_overall_statistics(test_df)


In [ ]:
# Data plotting

def plot_overview(df: pd.DataFrame, benchmark_name: str, output_dir: Path):
    """Plot overview: average latency and throughput across runs"""
    ops_df = df[df["operation"] != "CLEANUP"].copy()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Calculate per-run metrics
    run_metrics = []
    for run_num in sorted(ops_df["run_number"].unique()):
        run_df = ops_df[ops_df["run_number"] == run_num]
        duration = (
            run_df["absolute_timestamp_ms"].max() - run_df["absolute_timestamp_ms"].min()
        ) / 1000
        
        run_metrics.append({
            "run": run_num,
            "avg_latency_ms": run_df["latency_ms"].mean(),
            "throughput_ops_s": len(run_df) / duration if duration > 0 else 0
        })
    
    metrics_df = pd.DataFrame(run_metrics)

    # Convert run numbers to string for discrete x-axis
    metrics_df["run_str"] = metrics_df["run"].astype(str)
    
    # Plot 1: Average Latency
    ax1.bar(
        metrics_df["run_str"],
        metrics_df["avg_latency_ms"],
        color="steelblue",
        edgecolor="black",
        linewidth=1.5
    )
    ax1.axhline(
        y=metrics_df["avg_latency_ms"].mean(),
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Overall Avg: {metrics_df["avg_latency_ms"].mean():.1f} ms"
    )
    ax1.set_xlabel("Run Number", fontsize=12)
    ax1.set_ylabel("Average Latency (ms)", fontsize=12)
    ax1.set_title("Average Latency per Run", fontsize=14)
    ax1.legend()
    ax1.grid(True, axis="y", alpha=0.3)
    
    # Plot 2: Throughput
    ax2.bar(
        metrics_df["run_str"],
        metrics_df["throughput_ops_s"],
        color="darkgreen",
        edgecolor="black",
        linewidth=1.5
    )
    ax2.axhline(
        y=metrics_df["throughput_ops_s"].mean(),
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Overall Avg: {metrics_df["throughput_ops_s"].mean():.1f} ops/s"
    )
    ax2.set_xlabel("Run Number", fontsize=12)
    ax2.set_ylabel("Throughput (ops/s)", fontsize=12)
    ax2.set_title("Throughput per Run", fontsize=14)
    ax2.legend()
    ax2.grid(True, axis="y", alpha=0.3)
    
    plt.suptitle(
        f"{benchmark_name} - Performance Overview",
        fontsize=16,
        y=1.02
    )
    plt.tight_layout()
    plt.savefig(output_dir / "overview.png", dpi=300, bbox_inches="tight")
    plt.show()

def plot_percentiles_overall(
    df: pd.DataFrame, 
    benchmark_name: str, 
    output_dir: Path
):
    """Plot latency percentiles with bands across all runs"""
    
    percentiles = [1, 5, 50, 90, 95, 99, 99.9]
    
    # Calculate percentiles per run
    run_percentiles = []
    for run_num in df["run_number"].unique():
        run_df = df[df["run_number"] == run_num]
        latencies = run_df["latency_ms"]
        
        run_p = {"run": run_num}
        for p in percentiles:
            run_p[f"p{p}"] = np.percentile(latencies, p)
        run_percentiles.append(run_p)
    
    perc_df = pd.DataFrame(run_percentiles)
    
    # Calculate overall statistics
    overall_mean = [perc_df[f"p{p}"].mean() for p in percentiles]
    overall_std = [perc_df[f"p{p}"].std() for p in percentiles]
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    x = np.arange(len(percentiles))
    
    # Plot mean with error bands
    ax.plot(
        x, 
        overall_mean, 
        "o-", 
        linewidth=2, 
        markersize=8,
        color="steelblue",
        label="Mean across runs"
    )
    
    # Add ±1 std dev band
    ax.fill_between(
        x,
        np.array(overall_mean) - np.array(overall_std),
        np.array(overall_mean) + np.array(overall_std),
        alpha=0.3,
        color="steelblue",
        label="±1 Std Dev"
    )
    
    # Add individual run data points
    for _, row in perc_df.iterrows():
        run_vals = [row[f"p{p}"] for p in percentiles]
        ax.plot(
            x, 
            run_vals, 
            "o", 
            alpha=0.4, 
            markersize=5,
            color="gray"
        )
    
    ax.set_xticks(x)
    ax.set_xticklabels([f"p{p}" for p in percentiles], fontsize=11)
    ax.set_ylabel("Latency (ms)", fontsize=12)
    ax.set_xlabel("Percentile", fontsize=12)
    ax.set_title(
        f"{benchmark_name} - Latency Percentiles",
        fontsize=14
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add value labels
    for i, (mean_val, std_val) in enumerate(zip(overall_mean, overall_std)):
        ax.text(
            i, 
            mean_val, 
            f"{mean_val:.1f}",
            ha="center",
            va="bottom",
            fontsize=9
        )
    
    plt.tight_layout()
    plt.savefig(
        output_dir / "percentiles_overall.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

def plot_percentiles_by_client(
    df: pd.DataFrame,
    benchmark_name: str,
    output_dir: Path
):
    """Plot latency percentiles split by client"""
    
    percentiles = [1, 5, 50, 90, 95, 99, 99.9]
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    x = np.arange(len(percentiles))
    width = 0.2
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    
    for i, client_id in enumerate(sorted(df["client_id"].unique())):
        client_df = df[df["client_id"] == client_id]
        latencies = client_df["latency_ms"]
        
        values = [np.percentile(latencies, p) for p in percentiles]
        
        offset = width * (i - 1.5)
        ax.bar(
            x + offset,
            values,
            width,
            label=f"Client {region_mapping[str(client_id)]}",
            color=colors[i],
            edgecolor="black",
            linewidth=0.5
        )
    
    ax.set_xticks(x)
    ax.set_xticklabels([f"p{p}" for p in percentiles], fontsize=11)
    ax.set_ylabel("Latency (ms)", fontsize=12)
    ax.set_xlabel("Percentile", fontsize=12)
    ax.set_title(
        f"{benchmark_name} - Latency Percentiles by Client",
        fontsize=14
    )
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(
        output_dir / "percentiles_by_client.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

def plot_latency_over_time(
    df: pd.DataFrame,
    benchmark_name: str,
    output_dir: Path
):
    """Plot average latency over time in 1-second buckets"""
    
    # Use overall_run_ms which is relative time within each run
    df["time_bucket_s"] = (df["overall_run_ms"] // 1000).astype(int)
    
    # Group by run and time bucket
    bucketed = df.groupby(["run_number", "time_bucket_s"])[
        "latency_ms"
    ].mean().reset_index()
    
    fig, ax = plt.subplots(figsize=(16, 7))
    
    # Plot each run
    for run_num in sorted(bucketed["run_number"].unique()):
        run_data = bucketed[bucketed["run_number"] == run_num]
        ax.plot(
            run_data["time_bucket_s"],
            run_data["latency_ms"],
            alpha=0.3,
            linewidth=1,
            color="gray"
        )
    
    # Plot overall average across all runs
    overall_avg = bucketed.groupby("time_bucket_s")[
        "latency_ms"
    ].mean().reset_index()
    
    ax.plot(
        overall_avg["time_bucket_s"],
        overall_avg["latency_ms"],
        linewidth=2.5,
        color="steelblue",
        label="Average across all runs"
    )
    
    ax.set_xlabel("Time (ms)", fontsize=12)
    ax.set_ylabel("Average Latency (ms)", fontsize=12)
    ax.set_title(
        f"{benchmark_name} - Average Latency Over Time (1000 ms buckets)",
        fontsize=14
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(
        output_dir / "latency_over_time.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

benchmark_name = "trivial-raw-3000"
output_path = output_base_path / benchmark_name
output_path.mkdir(parents=True, exist_ok=True)

plot_overview(test_df, benchmark_name, output_path)
plot_percentiles_overall(test_df, benchmark_name, output_path)
plot_percentiles_by_client(test_df, benchmark_name, output_path)
plot_latency_over_time(test_df, benchmark_name, output_path)